In [ ]:
import numpy as np
import subprocess
import os


import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
from matplotlib.pyplot import figure, show, subplots
from matplotlib import pyplot as plt

In [ ]:
# get AGNI data

# find which molecules appear at concentration higher than 0.1%

file="../output/AGNI_atmospherecomposition"
data = np.genfromtxt(file,skip_header=1,delimiter=",")
datalength = len(data)
titles = np.genfromtxt(file,delimiter=",",dtype='str',skip_footer=datalength)

relevant_molecules = set()
relevant_molecules_index = set([0,1]) # don't delete fo2 and C/H indices

for i in range(len(data)):
    for j in range(2,len(data[i])):
        if data[i][j] > 0.001:
            relevant_molecules.add(titles[j])
            relevant_molecules_index.add(j)


# delete species that occur at >0.1%
cutlist = np.arange(51)
for i in list(relevant_molecules_index)[::-1]:
    cutlist = np.delete(cutlist,i,0)

data_cut = np.array(data)
titles_cut = np.array(titles)
for i in cutlist[::-1]:
    data_cut = np.delete(data_cut,i,1)
    titles_cut = np.delete(titles_cut,i,0)

titles_cut = titles_cut[2:]

In [ ]:
atmosphere_dict = {(i[0],i[1]): i[2:] for i in data_cut}
print(atmosphere_dict[(4.0,0.1)])
print(titles_cut)
print(atmosphere_dict.keys())

In [ ]:
def format_float(x):
    base, exp = f"{x:.2e}".split("e")
    exp = int(exp)
    return f"{base}d{exp}"
formatted = [format_float(v) for v in atmosphere_dict[(-4.0,0.1)]]


def ARCiS_file_generator(fo2, CH, AGNI_atmos, distance):
    """Generate planetary emission spectra using ARCiS"""

    formatted = [format_float(i) for i in AGNI_atmos]
    config_ARCiS_loc = "config/forward_model_ARCiS.in"
    atmosphere_key = 'fm_ARCiS' + str(fo2) + '_' + str(CH) + '_' + str(distance)
    save_location = f"/dataserver/users/formingworlds/borgmann/ARCiS_forwardmodels/{atmosphere_key}"
    command = (
        f"ARCiS {config_ARCiS_loc} -o {save_location} -s distance={distance}d0 "
        f"-s N2={formatted[0]} "
        f"-s CO={formatted[1]} "
        f"-s H2={formatted[2]} "
        f"-s CH4={formatted[3]} "
        f"-s H2S={formatted[4]} "
        f"-s S2=1d-50 "
        f"-s SO2={formatted[6]} "
        f"-s H2O={formatted[7]} "
        f"-s CO2={formatted[8]} "
        f"-s O2={formatted[9]} "
        f"-s H2SO4=1d-50 "
        f"-s SO={formatted[11]} "
        f"-s NH3={formatted[12]} "
        f"-s OCS={formatted[13]} "
    )
    print(repr(command))
      
    atmosphere_key_us = atmosphere_key.replace(".","_")
    prompt = (
        "ssh -n -T norma2;"
        "cd .. ;"
        f"tmux new-session -d -s '{atmosphere_key}'; "
        f"tmux send-keys -t {atmosphere_key_us}:0 'module unload ARCiS' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module load ARCiS' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module unload cfitsio' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module load cfitsio' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'export OMP_NUM_THREADS=8' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 '{command}' C-m"
        )
    subprocess.run(prompt, shell=True, check=True) 

In [ ]:
for keys in list(atmosphere_dict.keys()):
    for distance in [20]:
        ARCiS_file_generator(keys[0],keys[1],atmosphere_dict[(keys[0],keys[1])],distance)

In [ ]:
# sample plot

def spectrum_cutoff(bandcenter,bandflux):
    """Cut spectrum to only include 4-18.5 microns"""
    for i in range(len(bandcenter)):
        if bandcenter[i] > 4.0:
            lower_index = i-1
            break
    for j in range(len(bandcenter)):
        if bandcenter[j] > 18.5:
            upper_index = j+1
            break
    return bandcenter[lower_index:upper_index], bandflux[lower_index:upper_index]


done = []
data_location = "/dataserver/users/formingworlds/borgmann/ARCiS_forwardmodels/"
atmosphere_folders = os.listdir(data_location)
for i in atmosphere_folders:
    if os.path.isfile(data_location+i+"/emis"):
            done.append(i)

def sort_key(name):
    parts = name.replace("fm_ARCiS", "").split("_")
    fo2 = float(parts[0])
    CH = float(parts[1])
    dist = float(parts[2])
    return (fo2, CH, dist)

done = sorted(done, key=sort_key)
print(done)

            
#volatile_labels = np.genfromtxt(data_location+done[0]+"/retrieval",usecols=0,dtype='str')[1:13]

atmosphere_data = []
band = np.genfromtxt(data_location+'fm_ARCiS-4.0_0.1_5'+"/emis",usecols=0,skip_header=1)
flux = np.genfromtxt(data_location+'fm_ARCiS-4.0_0.1_5'+"/emis",usecols=1,skip_header=1)
band, flux = spectrum_cutoff(band,flux)

#atmosphere_data.append([[float(j) for j in done[i].split("_")],medians])

# make images of clean spectra
fig = figure(figsize=(8,6))
frame1 = fig.add_subplot(1,1,1)
frame1.plot(band,flux,label="ARCiS")
frame1.set_xlabel('wavelength (microns)')
frame1.set_ylabel('Janskys')
frame1.set_yscale('log')

AGNI_loc = '/Users/users/borgmann/Documents/masterproject/LIFEredoxsurvey/output/LIFEsim/clean_spectra/textfiles/clean_spectrum_-4.0_0.1_5_350.txt'
AGNI_band = np.genfromtxt(AGNI_loc,usecols=0)
AGNI_flux = np.genfromtxt(AGNI_loc,usecols=1)

AGNI_loc_old = '/dataserver/users/formingworlds/borgmann/old/databackup_020426/LIFEsim/clean_spectra/textfiles/clean_spectrum_-4.0_0.1_5.txt'
AGNI_band_old = np.genfromtxt(AGNI_loc_old,usecols=0)
AGNI_flux_old = np.genfromtxt(AGNI_loc_old,usecols=1)

frame1.plot(AGNI_band,AGNI_flux,label='AGNI (high res)')
frame1.plot(AGNI_band_old,AGNI_flux_old,label='AGNI (low res)')
frame1.legend()


    #frame1.set_xlim(4,18.5)
    #filename = '../output/LIFEsim/clean_spectra/images/'+ "clean" + agnifilename[4:-4] + "_" + str(distance) + "_" + str(int(integrationtime/60/60)) + '.png'

#atmosphere_dict = {(x,y,z): arr for [x,y,z], arr in atmosphere_data}


In [ ]:
# raster plot of ARCiS forward model spectra

data_path = "/dataserver/users/formingworlds/borgmann/ARCiS_forwardmodels"
files = [f for f in os.listdir(data_path)]

mpl.rcParams['figure.dpi'] = 800
rasterplot = figure(figsize=(8,6))

# setup custom order for rasterplot, since the subplots go left->right, up->down
# but we need to go down->up, left->right
custom_order = [9,5,1,10,6,2,11,7,3,12,8,4]

custom_count = 0
# do plots
for i in range(len(done)):
   if done[i].split("_")[-1] == '5':
        print(done[i])
        frame = rasterplot.add_subplot(3,4,custom_order[custom_count])
        bandcenter = np.genfromtxt(data_location+done[i]+"/emis",usecols=0,skip_header=1)
        bandflux = np.genfromtxt(data_location+done[i]+"/emis",usecols=1,skip_header=1)
        bandcenter, bandflux = spectrum_cutoff(bandcenter,bandflux)
        frame.plot(bandcenter,bandflux,linewidth=0.5)
        frame.set_yscale('log')
        frame.set_xlim(4,18.5)
        frame.set_ylim(10e-10,2.4122010236909366e-06)
        if custom_count in [1,2]:
            plt.tick_params(
                axis='both',      # both x and y axes
                which='both',     # major and minor ticks
                labelsize=6,      # ↓ font size of tick labels
                length=0,         # ↓ length of tick marks
                width=0.4,         # ↓ thickness of tick marks
                pad=0.8
            )
            frame.minorticks_off()
        elif custom_count == 0:
            plt.tick_params(
                axis='both',      # both x and y axes
                which='both',     # major and minor ticks
                labelsize=6,      # ↓ font size of tick labels
                length=0,         # ↓ length of tick marks
                width=0.4,         # ↓ thickness of tick marks
                pad=0.8
            )
            frame.minorticks_off()
            frame.set_xticks([])
            frame.set_xticklabels([])

        else:
            frame.set_yticklabels([])
            frame.set_xticklabels([])
            frame.set_xticks([])
            frame.set_yticks([])
            frame.minorticks_off()
        custom_count += 1
      

# add axes to the plot (I feel like I over-engineered this)
master_ax = rasterplot.add_axes([0.1, 0.1, 0.8, 0.8])
master_ax.patch.set_alpha(0)
master_ax.set_zorder(-10)

master_ax.spines['top'].set_visible(False)
master_ax.spines['right'].set_visible(False)
master_ax.spines['left'].set_linewidth(1.5)
master_ax.spines['bottom'].set_linewidth(1.5)

y_centers = [0.83, 0.50, 0.17] 
master_ax.set_ylim(0, 1)
master_ax.set_yticks(y_centers)
master_ax.set_yticklabels(["5", "1", "0.1"])

x_centers = [0.135, 0.365, 0.635, 0.865]
master_ax.set_xlim(0, 1)
master_ax.set_xticks(x_centers)
master_ax.set_xticklabels(["-4", "-1.3", "1.3", "4"])

master_ax.tick_params(labelsize=12)


rasterplot.supxlabel('Mantle $f$O$_2$, $\Delta$ IW')
rasterplot.supylabel('Mantle C/H mass ratio')
rasterplot.suptitle("AGNI spectra for varying mantle $f$O$_2$ and mantle H/C mass ratio")
rasterplot.show()
rasterplot.savefig('../output/figures/ARCiS_fm_raster.png')